# Adverse Drug Event (ADE) Data Exploration Notebook

This notebook demonstrates Domino Data Lab's capabilities for adverse drug event detection by providing comprehensive exploratory analysis of an FDA FAERS-inspired dataset. It showcases data exploration workflows including automated profiling, quality assessment, and pharmacovigilance-specific visualisations. The notebook serves as a reusable template for Sales Engineers demonstrating Domino to life sciences audiences.

In [ ]:
import io, os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime

## Connect to Data Source Connector

**Domino Data Source Connectors** provide a secure, driver-free way to access external data. Replace the snippet below with the Python snippet from your `adverse-drug-event-detection` Data Source.

In [ ]:
######################################################
#### Replace With Python Snippet From Data Source ####
######################################################
from domino.data_sources import DataSourceClient

object_store = DataSourceClient().get_datasource('adverse-drug-event-detection')
objects = object_store.list_objects()

## Load Raw ADE Report Data

In [ ]:
raw_filename  = 'raw_ade_reports.csv'
clean_filename = 'clean_ade_reports.csv'
datasource_name = 'adverse-drug-event-detection'

domino_working_dir  = os.environ.get('DOMINO_WORKING_DIR', '.')
domino_project_name = os.environ.get('DOMINO_PROJECT_NAME', 'my-local-project')
domino_dataset_dir  = f"{domino_working_dir.replace('code', 'data')}/{domino_project_name}"
domino_artifact_dir = domino_working_dir.replace('code', 'artifacts')

ds = DataSourceClient().get_datasource(datasource_name)
buf = io.BytesIO()
ds.download_fileobj(raw_filename, buf)
buf.seek(0)
raw_df = pd.read_csv(buf)
print(f'Loaded {len(raw_df):,} rows from {raw_filename}')

## Clean the Dataset

In [ ]:
clean_df = raw_df.dropna()
rows_removed = len(raw_df) - len(clean_df)
pct_removed  = 100 * rows_removed / len(raw_df)
print(f'Dropped {rows_removed:,} rows with missing data ({pct_removed:.2f}%)')
clean_df.head()

## Dataset Overview & Metadata

In [ ]:
print('=== DATASET OVERVIEW ===')
print(f'Shape: {clean_df.shape[0]:,} rows x {clean_df.shape[1]} columns')
print(f'Memory: {clean_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'Created: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

print('\n=== COLUMN INFORMATION ===')
column_info = pd.DataFrame({
    'Column':        clean_df.columns,
    'Data Type':     clean_df.dtypes,
    'Non-Null Count':clean_df.count(),
    'Null %':        (clean_df.isnull().sum() / len(clean_df) * 100).round(2),
    'Unique Values': clean_df.nunique()
})
print(column_info.to_string())

print('\n=== NUMERICAL FEATURES SUMMARY ===')
numeric_cols = clean_df.select_dtypes(include=[np.number]).columns
print(clean_df[numeric_cols].describe().round(3))

print('\n=== CATEGORICAL FEATURES SUMMARY ===')
cat_cols = clean_df.select_dtypes(include=['object']).columns
for col in cat_cols:
    vc = clean_df[col].value_counts()
    print(f'\n{col} ({clean_df[col].nunique()} unique):')
    for val, cnt in vc.head(5).items():
        print(f'  {val}: {cnt:,} ({cnt/len(clean_df)*100:.1f}%)')

## Target Variable: Seriousness Distribution

In [ ]:
print('=== TARGET VARIABLE: Serious ===')
serious_dist = clean_df['Serious'].value_counts()
for val, cnt in serious_dist.items():
    label = 'Serious' if val == 1 else 'Non-Serious'
    print(f'  {label}: {cnt:,} ({cnt/len(clean_df)*100:.1f}%)')
imbalance = serious_dist.max() / serious_dist.min()
print(f'\nClass imbalance ratio: {imbalance:.2f}:1')

## Visualisation 1: Seriousness Rate by Drug Class

In [ ]:
serious_by_class = clean_df.groupby('drug_class')['Serious'].agg(['mean', 'count'])
serious_by_class['serious_rate_pct'] = serious_by_class['mean'] * 100
serious_by_class = serious_by_class.sort_values('serious_rate_pct', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.9, len(serious_by_class)))
bars = ax.bar(range(len(serious_by_class)), serious_by_class['serious_rate_pct'], color=colors, edgecolor='white')
for i, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{serious_by_class["serious_rate_pct"].iloc[i]:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_xticks(range(len(serious_by_class)))
ax.set_xticklabels(serious_by_class.index, rotation=35, ha='right')
ax.set_ylabel('Seriousness Rate (%)', fontsize=12)
ax.set_title('Seriousness Rate by Drug Class', fontsize=15, fontweight='bold', pad=15)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{domino_artifact_dir}/seriousness_by_drug_class.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Highest seriousness: {serious_by_class.index[0]} ({serious_by_class["serious_rate_pct"].iloc[0]:.1f}%)')

## Visualisation 2: Age Distribution by Seriousness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (val, label, color) in zip(axes, [(0, 'Non-Serious', '#2E7D32'), (1, 'Serious', '#B71C1C')]):
    subset = clean_df[clean_df['Serious'] == val]['age']
    ax.hist(subset, bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(subset.mean(), color='black', linestyle='--', linewidth=2, label=f'Mean: {subset.mean():.1f}')
    ax.set_title(f'Age Distribution — {label}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Patient Age')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{domino_artifact_dir}/age_distribution_by_seriousness.png', dpi=300, bbox_inches='tight')
plt.show()

## Visualisation 3: Seriousness Rate by Reaction Category

In [ ]:
serious_by_reaction = clean_df.groupby('reaction_category')['Serious'].mean().sort_values(ascending=True) * 100

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#B71C1C' if v > 50 else '#1B5E82' for v in serious_by_reaction]
bars = ax.barh(serious_by_reaction.index, serious_by_reaction.values, color=colors, alpha=0.8)
for bar in bars:
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2.,
            f'{bar.get_width():.1f}%', va='center', fontsize=10)
ax.set_xlabel('Seriousness Rate (%)', fontsize=12)
ax.set_title('Seriousness Rate by Reaction Category', fontsize=14, fontweight='bold', pad=15)
ax.axvline(50, color='gray', linestyle='--', alpha=0.5, label='50% threshold')
ax.legend()
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{domino_artifact_dir}/seriousness_by_reaction.png', dpi=300, bbox_inches='tight')
plt.show()

## Visualisation 4: Correlation Heatmap

In [ ]:
numeric_features = ['age', 'weight_kg', 'dose_mg', 'duration_days', 'concurrent_meds',
                    'time_to_onset_days', 'comorbidity_count', 'prior_ade', 'Serious']
corr_matrix = clean_df[numeric_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_features)))
ax.set_yticks(range(len(numeric_features)))
ax.set_xticklabels(numeric_features, rotation=45, ha='right')
ax.set_yticklabels(numeric_features)
plt.colorbar(im, ax=ax, label='Correlation Coefficient')
for i in range(len(numeric_features)):
    for j in range(len(numeric_features)):
        val = corr_matrix.iloc[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                color='white' if abs(val) > 0.5 else 'black', fontsize=8)
ax.set_title('Feature Correlation Heatmap — ADE Reports', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{domino_artifact_dir}/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n=== Strongest Correlations with Seriousness ===')
serious_corr = corr_matrix['Serious'].drop('Serious').sort_values(key=abs, ascending=False)
print(serious_corr)

## Visualisation 5: Concurrent Medications vs Seriousness

In [ ]:
polypharmacy = clean_df.groupby('concurrent_meds')['Serious'].agg(['mean', 'count'])
polypharmacy['serious_rate_pct'] = polypharmacy['mean'] * 100

fig, ax1 = plt.subplots(figsize=(12, 6))
ax2 = ax1.twinx()
ax1.bar(polypharmacy.index, polypharmacy['count'], color='#1B5E82', alpha=0.4, label='Report Count')
ax2.plot(polypharmacy.index, polypharmacy['serious_rate_pct'], 'o-',
         color='#B71C1C', linewidth=2, markersize=7, label='Seriousness Rate')
ax1.set_xlabel('Number of Concurrent Medications', fontsize=12)
ax1.set_ylabel('Report Count', fontsize=12, color='#1B5E82')
ax2.set_ylabel('Seriousness Rate (%)', fontsize=12, color='#B71C1C')
ax1.set_title('Polypharmacy Effect on ADE Seriousness', fontsize=14, fontweight='bold', pad=15)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{domino_artifact_dir}/polypharmacy_seriousness.png', dpi=300, bbox_inches='tight')
plt.show()

## Data Quality Report

In [ ]:
print('=== DATA QUALITY REPORT ===')
print(f'No missing values (cleaned dataset)')
print(f'Binary target variable (Serious: 0 or 1)')
print(f'{len(clean_df[clean_df.duplicated()]):,} duplicate rows found')

outlier_summary = {}
for col in clean_df.select_dtypes(include=[np.number]).columns:
    if col not in ['Serious', 'prior_ade', 'report_id']:
        Q1, Q3 = clean_df[col].quantile(0.25), clean_df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((clean_df[col] < Q1 - 1.5*IQR) | (clean_df[col] > Q3 + 1.5*IQR)).sum()
        if outliers > 0:
            outlier_summary[col] = outliers

if outlier_summary:
    print('\nPotential outliers (IQR method):')
    for col, cnt in outlier_summary.items():
        print(f'  {col}: {cnt:,} ({cnt/len(clean_df)*100:.1f}%)')

## Save Clean Dataset to Domino Dataset

In [ ]:
os.makedirs(domino_dataset_dir, exist_ok=True)
clean_df.to_csv(f'{domino_dataset_dir}/{clean_filename}')
print(f'Saved clean dataset to {domino_dataset_dir}/{clean_filename}')